# IntelliDoc AI — Notebook 3: LayoutLMv3 Fine-Tuning

**Purpose:** Fine-tune the pretrained LayoutLMv3 model on the FUNSD training set and evaluate it on the test set. Compare the results to the baseline from Notebook 2.

**Research meaning:** The improvement from Notebook 2 baseline → this notebook's scores shows the value of task-specific fine-tuning.

---
> **Prerequisites:** Run Notebook 1 and Notebook 2 first.
>
> **Runtime:** This notebook takes approximately **20–40 minutes** on a Colab T4 GPU.

## Step 1: Mount Drive, Set Paths, Install Libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
FUNSD_ROOT    = '/content/drive/MyDrive/funsd'
TRAIN_DIR     = os.path.join(FUNSD_ROOT, 'training_data')
TEST_DIR      = os.path.join(FUNSD_ROOT, 'testing_data')
CHECKPOINT_DIR = '/content/drive/MyDrive/intellidoc_checkpoints/layoutlmv3_funsd'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

for path in [TRAIN_DIR, TEST_DIR]:
    if not os.path.exists(path):
        raise FileNotFoundError(f'Missing: {path}')
print('Paths verified. Checkpoint will be saved to:', CHECKPOINT_DIR)

In [ ]:
!pip install -q transformers datasets seqeval Pillow timm accelerate
print('Libraries ready.')

## Step 2: Load and Preprocess FUNSD (Both Splits)
Same preprocessing as Notebook 2, but now we also load the **training set** for fine-tuning.

In [ ]:
import json
import glob
from PIL import Image

LABEL2ID = {'other': 0, 'question': 1, 'answer': 2, 'header': 3}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)

def normalize_bbox(bbox, width, height):
    x0, y0, x1, y1 = bbox
    return [
        int(1000 * x0 / width),
        int(1000 * y0 / height),
        int(1000 * x1 / width),
        int(1000 * y1 / height),
    ]

def load_funsd_for_layoutlm(split_dir):
    ann_dir = os.path.join(split_dir, 'annotations')
    img_dir = os.path.join(split_dir, 'images')
    samples = []
    for ann_file in sorted(glob.glob(os.path.join(ann_dir, '*.json'))):
        fname = os.path.basename(ann_file).replace('.json', '')
        img_path = os.path.join(img_dir, fname + '.png')
        if not os.path.exists(img_path):
            continue
        image = Image.open(img_path).convert('RGB')
        width, height = image.size
        with open(ann_file) as f:
            ann = json.load(f)
        words, bboxes, labels = [], [], []
        for item in ann.get('form', []):
            label_id = LABEL2ID.get(item.get('label', 'other'), 0)
            for word_info in item.get('words', []):
                word = word_info.get('text', '').strip()
                box  = word_info.get('box', [0, 0, 0, 0])
                if word:
                    words.append(word)
                    bboxes.append(normalize_bbox(box, width, height))
                    labels.append(label_id)
        samples.append({'image': image, 'words': words,
                         'bboxes': bboxes, 'labels': labels, 'filename': fname})
    return samples

print('Loading datasets...')
train_samples = load_funsd_for_layoutlm(TRAIN_DIR)
test_samples  = load_funsd_for_layoutlm(TEST_DIR)
print(f'Train: {len(train_samples)} docs  |  Test: {len(test_samples)} docs')

## Step 3: Build PyTorch Datasets
Convert our raw samples into a format PyTorch can use for training.

In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import LayoutLMv3Processor

MODEL_NAME = 'microsoft/layoutlmv3-base'
processor  = LayoutLMv3Processor.from_pretrained(MODEL_NAME, apply_ocr=False)

class FUNSDDataset(Dataset):
    def __init__(self, samples, processor, max_length=512):
        self.samples    = samples
        self.processor  = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        encoding = self.processor(
            sample['image'],
            sample['words'],
            boxes=sample['bboxes'],
            word_labels=sample['labels'],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        # Remove batch dimension added by processor
        return {k: v.squeeze(0) for k, v in encoding.items()}

train_dataset = FUNSDDataset(train_samples, processor)
test_dataset  = FUNSDDataset(test_samples,  processor)
print(f'Train dataset size: {len(train_dataset)}')
print(f'Test  dataset size: {len(test_dataset)}')

## Step 4: Load Model and Configure Training
Set up the LayoutLMv3 model with a token classification head on top.

In [ ]:
from transformers import LayoutLMv3ForTokenClassification, TrainingArguments, Trainer
from seqeval.metrics import f1_score, precision_score, recall_score, classification_report
import numpy as np

model = LayoutLMv3ForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True
)

print('Model loaded. Classification head initialized.')
print(f'Number of parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M')

In [ ]:
# ---- Training Configuration ----
# These settings are tuned for FUNSD on a free Colab T4 GPU.
# Training will run for ~20-40 minutes.
training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=10,           # Number of full passes through the training data
    per_device_train_batch_size=2, # Documents processed at once (small = fits in GPU memory)
    per_device_eval_batch_size=2,
    learning_rate=1e-5,            # How fast the model learns (small = careful updates)
    warmup_ratio=0.1,              # Gradually ramp up learning rate at start
    weight_decay=0.01,             # Regularization to prevent overfitting
    eval_strategy='epoch',         # Evaluate after each epoch
    save_strategy='epoch',         # Save checkpoint after each epoch
    load_best_model_at_end=True,   # Keep the best checkpoint at the end
    metric_for_best_model='f1',    # Select best model based on F1 score
    greater_is_better=True,
    logging_steps=10,
    fp16=True,                     # Use mixed precision for faster GPU training
    report_to='none',              # Disable wandb / other loggers
)
print('Training configuration set.')

In [ ]:
def compute_metrics(eval_pred):
    """Called after each epoch to compute F1, Precision, Recall."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    true_labels = []
    pred_labels = []

    for pred_row, label_row in zip(predictions, labels):
        true_seq, pred_seq = [], []
        for p, l in zip(pred_row, label_row):
            if l != -100:  # -100 is the ignore index for padding tokens
                true_seq.append(ID2LABEL[l])
                pred_seq.append(ID2LABEL[p])
        true_labels.append(true_seq)
        pred_labels.append(pred_seq)

    return {
        'f1'       : f1_score(true_labels, pred_labels),
        'precision': precision_score(true_labels, pred_labels),
        'recall'   : recall_score(true_labels, pred_labels),
    }

## Step 5: Start Fine-Tuning
This runs the actual training. It will print loss and F1 after each epoch.

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print('Starting fine-tuning...')
print('This will take approximately 20-40 minutes on a T4 GPU.')
print()
trainer.train()

## Step 6: Final Evaluation on Test Set
Run the fine-tuned model on the test set and print the final scores.

In [ ]:
predictions_output = trainer.predict(test_dataset)
logits = predictions_output.predictions
label_ids = predictions_output.label_ids
preds = np.argmax(logits, axis=-1)

true_labels, pred_labels = [], []
for pred_row, label_row in zip(preds, label_ids):
    true_seq, pred_seq = [], []
    for p, l in zip(pred_row, label_row):
        if l != -100:
            true_seq.append(ID2LABEL[l])
            pred_seq.append(ID2LABEL[p])
    true_labels.append(true_seq)
    pred_labels.append(pred_seq)

report = classification_report(true_labels, pred_labels, digits=4)

print('=' * 60)
print('INTELLIDOC AI — LAYOUTLMV3 FINE-TUNED RESULTS')
print('Model  : microsoft/layoutlmv3-base  (FINE-TUNED on FUNSD)')
print('Dataset: FUNSD Test Set')
print('=' * 60)
print(report)
print()
print(f'Final F1 Score  : {f1_score(true_labels, pred_labels):.4f}')
print(f'Final Precision : {precision_score(true_labels, pred_labels):.4f}')
print(f'Final Recall    : {recall_score(true_labels, pred_labels):.4f}')
print('=' * 60)

## Step 7: Save the Fine-Tuned Model Checkpoint
Save the best model to Google Drive for later use in the IntelliDoc runtime app.

In [ ]:
FINAL_SAVE_PATH = os.path.join(CHECKPOINT_DIR, 'final_model')
trainer.save_model(FINAL_SAVE_PATH)
processor.save_pretrained(FINAL_SAVE_PATH)

print(f'Model saved to: {FINAL_SAVE_PATH}')
print()
print('Next step: Download this folder from Google Drive and copy it to:')
print('  IntelliDoc_AI/offline_research/checkpoints/layoutlmv3_funsd/')
print('This checkpoint will then be used by the IntelliDoc runtime application.')

## ✅ Fine-Tuning Complete

### Research Summary — Fill in After Running Both Notebooks

| Metric | Baseline (Notebook 2) | Fine-Tuned (This Notebook) | Improvement |
|--------|-----------------------|---------------------------|-------------|
| Overall F1 | ___ | ___ | +___ |
| Question F1 | ___ | ___ | +___ |
| Answer F1 | ___ | ___ | +___ |
| Header F1 | ___ | ___ | +___ |

This table becomes a key result in your IntelliDoc AI research thesis.

### Next Steps
1. Download the saved checkpoint from Google Drive
2. Place it in `IntelliDoc_AI/offline_research/checkpoints/layoutlmv3_funsd/`
3. The runtime application will load it for live document understanding